In [1]:
from pathlib import Path
import json
import geopandas as gpd
import pandas as pd
import polars as pl
from requests import head
from shapely.geometry import shape

In [2]:
silver_dir = Path('../data/silver/buildings')
gold_dir = Path('../data/gold')
gold_dir.mkdir(parents=True, exist_ok=True)
adm1_path = Path('../data/boundaries/geoBoundaries-IDN-ADM1-provinces.geojson')
adm2_path = Path('../data/boundaries/geoBoundaries-IDN-ADM2-districts.geojson')

silver_glob = (silver_dir / '*.parquet').as_posix()
all_silver = pl.scan_parquet(silver_glob)

In [3]:
building_count_quadkey = (
    all_silver
    .group_by('quadkey')
    .agg(
        pl.len().alias('building_count')
    )
    .collect()
)

building_count_quadkey.write_parquet(f'{gold_dir}/building_count_quadkey.parquet')

In [4]:
building_count_country = (
    all_silver
    .select(
        pl.col('country').first(),
        pl.len().alias('building_count')
    )
    .collect()
)

building_count_country.write_parquet(f'{gold_dir}/building_count_country.parquet')

In [5]:
silver_files = sorted(silver_dir.glob('*.parquet'))
provinces = (
    gpd.read_file(adm1_path)
    [['shapeName', 'shapeISO', 'geometry']]
    .rename(columns={'shapeName': 'province_name', 'shapeISO': 'province_code'})
)
districts = (
    gpd.read_file(adm2_path)
    [['shapeName', 'shapeID', 'geometry']]
    .rename(columns={'shapeName': 'district_name', 'shapeID': 'district_code'})
)

In [6]:
def load_building_points(silver_path: Path, crs):
    frame = (
        pl.read_parquet(silver_path)
        .select('silver_record_id', 'quadkey', 'source_file', 'geometry_geojson')
        .to_pandas()
    )
    frame['building_point'] = [
        shape(json.loads(geometry_geojson)).representative_point()
        for geometry_geojson in frame['geometry_geojson']
    ]
    return gpd.GeoDataFrame(
        frame[['silver_record_id', 'quadkey', 'source_file', 'building_point']],
        geometry='building_point',
        crs=crs,
    )

def summarize_spatial_join(building_points, boundaries, name_columns):
    joined = (
        gpd.sjoin(
            building_points,
            boundaries[name_columns + ['geometry']],
            how='left',
            predicate='within',
        )
        .drop(columns='index_right', errors='ignore')
    )
    counts = (
        joined.dropna(subset=[name_columns[0]])
        .groupby(name_columns)
        .size()
        .reset_index(name='building_count')
    )
    unmatched = joined.loc[
        joined[name_columns[0]].isna(),
        ['silver_record_id', 'quadkey', 'source_file', 'building_point'],
    ].copy()
    return counts, unmatched

print(f'Loaded {len(silver_files)} silver files')
print(provinces[['province_name', 'province_code']].head())
print(districts[['district_name', 'district_code']].head())

Loaded 601 silver files
        province_name province_code
0                Bali         ID-BA
1  West Nusa Tenggara         ID-NB
2              Banten         ID-BT
3        Central Java         ID-JT
4           West Java         ID-JB
     district_name            district_code
0       Aceh Barat  22746128B65593111718524
1  Aceh Barat Daya    22746128B227561513795
2       Aceh Besar  22746128B90547447297479
3        Aceh Jaya  22746128B21424895456858
4     Aceh Selatan  22746128B12428400263117


In [7]:
province_count_frames = []
province_unmatched_frames = []

for i, silver_path in enumerate(silver_files, start=1):
    building_points = load_building_points(silver_path, provinces.crs)
    province_counts, province_unmatched = summarize_spatial_join(
        building_points,
        provinces,
        ['province_name', 'province_code'],
    )
    province_count_frames.append(province_counts)
    province_unmatched_frames.append(province_unmatched)

    if i % 50 == 0 or i == len(silver_files):
        print(f'Processed {i}/{len(silver_files)} silver files for provinces')

province_counts = (
    pd.concat(province_count_frames, ignore_index=True)
    .groupby(['province_name', 'province_code'], as_index=False)['building_count']
    .sum()
    .sort_values(['province_name', 'province_code'])
)
province_unmatched = pd.concat(province_unmatched_frames, ignore_index=True)

pl.from_pandas(province_counts).write_parquet(
    gold_dir / 'building_counts_by_province.parquet'
)
gpd.GeoDataFrame(
    province_unmatched,
    geometry='building_point',
    crs=provinces.crs,
).to_parquet(
    gold_dir / 'building_counts_by_province_unmatched.parquet'
)

print(gold_dir / 'building_counts_by_province.parquet')
print(gold_dir / 'building_counts_by_province_unmatched.parquet')

Processed 50/601 silver files for provinces
Processed 100/601 silver files for provinces
Processed 150/601 silver files for provinces
Processed 200/601 silver files for provinces
Processed 250/601 silver files for provinces
Processed 300/601 silver files for provinces
Processed 350/601 silver files for provinces
Processed 400/601 silver files for provinces
Processed 450/601 silver files for provinces
Processed 500/601 silver files for provinces
Processed 550/601 silver files for provinces
Processed 600/601 silver files for provinces
Processed 601/601 silver files for provinces
../data/gold/building_counts_by_province.parquet
../data/gold/building_counts_by_province_unmatched.parquet


In [8]:
district_count_frames = []
district_unmatched_frames = []

for i, silver_path in enumerate(silver_files, start=1):
    building_points = load_building_points(silver_path, districts.crs)
    district_counts, district_unmatched = summarize_spatial_join(
        building_points,
        districts,
        ['district_name', 'district_code'],
    )
    district_count_frames.append(district_counts)
    district_unmatched_frames.append(district_unmatched)

    if i % 50 == 0 or i == len(silver_files):
        print(f'Processed {i}/{len(silver_files)} silver files for districts')

district_counts = (
    pd.concat(district_count_frames, ignore_index=True)
    .groupby(['district_name', 'district_code'], as_index=False)['building_count']
    .sum()
    .sort_values(['district_name', 'district_code'])
)
district_unmatched = pd.concat(district_unmatched_frames, ignore_index=True)

pl.from_pandas(district_counts).write_parquet(
    gold_dir / 'building_counts_by_district.parquet'
)
gpd.GeoDataFrame(
    district_unmatched,
    geometry='building_point',
    crs=districts.crs,
).to_parquet(
    gold_dir / 'building_counts_by_district_unmatched.parquet'
)

print(gold_dir / 'building_counts_by_district.parquet')
print(gold_dir / 'building_counts_by_district_unmatched.parquet')

Processed 50/601 silver files for districts
Processed 100/601 silver files for districts
Processed 150/601 silver files for districts
Processed 200/601 silver files for districts
Processed 250/601 silver files for districts
Processed 300/601 silver files for districts
Processed 350/601 silver files for districts
Processed 400/601 silver files for districts
Processed 450/601 silver files for districts
Processed 500/601 silver files for districts
Processed 550/601 silver files for districts
Processed 600/601 silver files for districts
Processed 601/601 silver files for districts
../data/gold/building_counts_by_district.parquet
../data/gold/building_counts_by_district_unmatched.parquet


In [9]:
province_unmatched_gdf = gpd.read_parquet(
    gold_dir / 'building_counts_by_province_unmatched.parquet'
)
district_unmatched_gdf = gpd.read_parquet(
    gold_dir / 'building_counts_by_district_unmatched.parquet'
)

province_unmatched_metric = province_unmatched_gdf.to_crs('EPSG:3857')
district_unmatched_metric = district_unmatched_gdf.to_crs('EPSG:3857')
provinces_metric = provinces.to_crs('EPSG:3857')
districts_metric = districts.to_crs('EPSG:3857')

def nearest_boundary_join(unmatched_gdf, boundary_gdf, keep_columns):
    joined = (
        gpd.sjoin_nearest(
            unmatched_gdf,
            boundary_gdf[keep_columns + ['geometry']],
            how='left',
            distance_col='distance_m',
        )
    )

    joined = (
        joined
        .sort_values(['silver_record_id', 'distance_m', *keep_columns], kind='stable')
        .drop_duplicates(subset='silver_record_id', keep='first')
        .drop(columns='index_right', errors='ignore')
        .to_crs('EPSG:4326')
        .reset_index(drop=True)
    )

    return joined

print(f'Province unmatched rows: {len(province_unmatched_gdf):,}')
print(f'District unmatched rows: {len(district_unmatched_gdf):,}')

Province unmatched rows: 138,989
District unmatched rows: 108,795


In [10]:
nearest_province_for_province_unmatched = nearest_boundary_join(
    province_unmatched_metric,
    provinces_metric,
    ['province_name', 'province_code'],
)

nearest_province_for_province_unmatched.to_parquet(
    gold_dir / 'building_counts_by_province_unmatched_nearest_adm1.parquet'
)

print(gold_dir / 'building_counts_by_province_unmatched_nearest_adm1.parquet')
nearest_province_for_province_unmatched[[
    'silver_record_id',
    'quadkey',
    'source_file',
    'province_name',
    'province_code',
    'distance_m',
]].head()

../data/gold/building_counts_by_province_unmatched_nearest_adm1.parquet


,silver_record_id,quadkey,source_file,province_name,province_code,distance_m
0,Indonesia_132220333_2026-02-23.csv.gz:1067,132220333,Indonesia_132220333_2026-02-23.csv.gz,Aceh,ID-AC,471.329107
1,Indonesia_132220333_2026-02-23.csv.gz:10770,132220333,Indonesia_132220333_2026-02-23.csv.gz,Aceh,ID-AC,23.621750
2,Indonesia_132220333_2026-02-23.csv.gz:1082,132220333,Indonesia_132220333_2026-02-23.csv.gz,Aceh,ID-AC,7.216306
3,Indonesia_132220333_2026-02-23.csv.gz:12367,132220333,Indonesia_132220333_2026-02-23.csv.gz,Aceh,ID-AC,633.610732
4,Indonesia_132220333_2026-02-23.csv.gz:12460,132220333,Indonesia_132220333_2026-02-23.csv.gz,Aceh,ID-AC,1257.401181


In [11]:
nearest_district_for_district_unmatched = nearest_boundary_join(
    district_unmatched_metric,
    districts_metric,
    ['district_name', 'district_code'],
)

nearest_district_for_district_unmatched.to_parquet(
    gold_dir / 'building_counts_by_district_unmatched_nearest_adm2.parquet'
)

print(gold_dir / 'building_counts_by_district_unmatched_nearest_adm2.parquet')
nearest_district_for_district_unmatched[[
    'silver_record_id',
    'quadkey',
    'source_file',
    'district_name',
    'district_code',
    'distance_m',
]].head()

../data/gold/building_counts_by_district_unmatched_nearest_adm2.parquet


,silver_record_id,quadkey,source_file,district_name,district_code,distance_m
0,Indonesia_132220333_2026-02-23.csv.gz:10440,132220333,Indonesia_132220333_2026-02-23.csv.gz,Kota Sabang,22746128B78398648082643,10.488121
1,Indonesia_132220333_2026-02-23.csv.gz:1082,132220333,Indonesia_132220333_2026-02-23.csv.gz,Kota Sabang,22746128B78398648082643,10.702380
2,Indonesia_132220333_2026-02-23.csv.gz:12083,132220333,Indonesia_132220333_2026-02-23.csv.gz,Aceh Besar,22746128B90547447297479,3.572016
3,Indonesia_132220333_2026-02-23.csv.gz:12270,132220333,Indonesia_132220333_2026-02-23.csv.gz,Aceh Besar,22746128B90547447297479,23.951119
4,Indonesia_132220333_2026-02-23.csv.gz:12383,132220333,Indonesia_132220333_2026-02-23.csv.gz,Kota Sabang,22746128B78398648082643,4.614717


In [12]:
prov_match = pl.from_pandas(province_counts)
prov_unmatch = (
    pl.read_parquet('../data/gold/building_counts_by_province_unmatched_nearest_adm1.parquet')
    .select('province_name', 'province_code')
    .group_by(['province_name', 'province_code'])
    .len()
    .rename({'len': 'building_count'})
)

prov_final = (
    prov_match
    .join(
        prov_unmatch,
        on=['province_name', 'province_code'],
        how='full',
        suffix='_unmatched',
    )
    .with_columns(
        pl.col('building_count').fill_null(0),
        pl.col('building_count_unmatched').fill_null(0),
    )
    .with_columns(
        (pl.col('building_count') + pl.col('building_count_unmatched')).alias('building_count'),
        pl.coalesce('province_name', 'province_name_unmatched').alias('province_name'),
        pl.coalesce('province_code', 'province_code_unmatched').alias('province_code'),
    )
    .select(['province_name', 'province_code', 'building_count'])
    .sort(['province_name', 'province_code'])
)

prov_final.write_parquet(gold_dir / 'building_counts_by_province.parquet')

/tmp/ipykernel_335098/3598625614.py:3: UserWarning: Extension type 'geoarrow.wkb' is not registered; loading as its storage type.

To avoid this warning, register the extension type or set environment variable 'POLARS_UNKNOWN_EXTENSION_TYPE_BEHAVIOR' to 'load_as_storage' or 'load_as_extension'.

In Polars 2.0, the default behavior will change to 'load_as_extension'.
  pl.read_parquet('../data/gold/building_counts_by_province_unmatched_nearest_adm1.parquet')


In [13]:
dis_match = pl.from_pandas(district_counts)
dis_unmatch = (
    pl.read_parquet('../data/gold/building_counts_by_district_unmatched_nearest_adm2.parquet')
    .select('district_name', 'district_code')
    .group_by(['district_name', 'district_code'])
    .len()
    .rename({'len': 'building_count'})
)

dis_final = (
    dis_match
    .join(
        dis_unmatch,
        on=['district_name', 'district_code'],
        how='full',
        suffix='_unmatched',
    )
    .with_columns(
        pl.col('building_count').fill_null(0),
        pl.col('building_count_unmatched').fill_null(0),
    )
    .with_columns(
        (pl.col('building_count') + pl.col('building_count_unmatched')).alias('building_count'),
        pl.coalesce('district_name', 'district_name_unmatched').alias('district_name'),
        pl.coalesce('district_code', 'district_code_unmatched').alias('district_code'),
    )
    .select(['district_name', 'district_code', 'building_count'])
    .sort(['district_name', 'district_code'])
)

dis_final.write_parquet(gold_dir / 'building_counts_by_district.parquet')